In [1]:
import gc, os, numpy as np, pandas as pd, matplotlib.pyplot as plt

os.environ["TOKENIZERS_PARALLELISM"] = "false"

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

# device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
device = torch.device("cpu")
df = pd.read_csv("../shopee-product-matching/train.csv")

device

device(type='cpu')

In [2]:
target_map = df.groupby('label_group')['posting_id'].apply(list).to_dict()
df['target'] = df['label_group'].map(target_map)

In [3]:
def calc_mean_f1(y_true, y_pred):
    f1_scores = []
    for true, pred in zip(y_true, y_pred):
        t_set = set(true)
        p_set = set(pred)
        intersection = len(p_set & t_set)
        f1 = (2.0 * intersection) / (len(t_set) + len(p_set)) if (len(t_set) + len(p_set)) > 0 else 0.0
        f1_scores.append(f1)
    return np.mean(f1_scores)

df[['posting_id', 'title', 'label_group']].head()

,posting_id,title,label_group
0,train_129225211,Paper Bag Victoria Secret,249114794
1,train_3386243561,"Double Tape 3M VHB 12 mm x 4,5 m ORIGINAL / DO...",2937985045
2,train_2288590299,Maling TTS Canned Pork Luncheon Meat 397 gr,2395904891
3,train_2406599165,Daster Batik Lengan pendek - Motif Acak / Camp...,4093212188
4,train_3369186413,Nescafe \xc3\x89clair Latte 220ml,3648931069


In [4]:
tfidf= TfidfVectorizer(
    analyzer="char_wb",
    ngram_range = (2, 6),
    max_features=35000,
    binary=True
)

text_matrix = tfidf.fit_transform(df['title'])

knn = NearestNeighbors(n_neighbors=50, metric='cosine', n_jobs=-1)
knn.fit(text_matrix)
distances, indices = knn.kneighbors(text_matrix)

thresholds = np.arange(0.3, 0.75, 0.05)
tfidf_f1_scores = []

for thresh in thresholds:
    preds = []
    for i in range(len(df)):
        valid_mask = (1. - distances[i]) >= thresh
        matched_idx = indices[i][valid_mask]
        preds.append(df['posting_id'].iloc[matched_idx].tolist())

    scores = calc_mean_f1(df['target'], preds)
    tfidf_f1_scores.append(scores)

best_tfidf_idx = int(np.argmax(tfidf_f1_scores))
best_tfidf_thresh = thresholds[best_tfidf_idx]

print("Best threshold: ", best_tfidf_thresh)
print("Baseline F1 Score: ", tfidf_f1_scores[best_tfidf_idx])


df['tfidf_predictions'] = [
    df['posting_id'].iloc[indices[i][(1.0 - distances[i]) >= best_tfidf_thresh]].tolist()
    for i in range(len(df))
]

Best threshold:  0.6499999999999999
Baseline F1 Score:  0.6347930260199842


In [ ]:
# 1. Create a local directory
!mkdir -p local_minilm

# 2. Use Mac's native 'curl' to download the files, bypassing Python's network stack
print("Downloading config...")
!curl -s -L -o local_minilm/config.json https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/config.json

print("Downloading tokenizer...")
!curl -s -L -o local_minilm/tokenizer.json https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/tokenizer.json
!curl -s -L -o local_minilm/tokenizer_config.json https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/tokenizer_config.json
!curl -s -L -o local_minilm/special_tokens_map.json https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/special_tokens_map.json
!curl -s -L -o local_minilm/vocab.txt https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/vocab.txt

print("Downloading model weights (this may take a minute or two)...")
!curl -L -o local_minilm/pytorch_model.bin https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/pytorch_model.bin

print("All files downloaded successfully!")

In [ ]:

model_name = "./local_minilm"

tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
model = AutoModel.from_pretrained(model_name, local_files_only=True)
model = model.to(device)
model.eval()

class TextDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts
    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx]
    
def extract_dense_embeddings(texts, batch_size = 64):
    loader = DataLoader(TextDataset(texts), batch_size=batch_size, shuffle=False)
    embedding_list=[]
    with torch.no_grad():
        for batch in tqdm(loader, desc='encoding_titles'):
            inputs = tokenizer(
                batch,
                padding=True, 
                truncation=True,
                max_length=64,
                return_tensors = "pt"
            ).to(device)
            
            outputs = model(**inputs)

            mask = inputs['attention_mask'].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()

            sum_embeddings=torch.sum(outputs.last_hidden_state * mask, 1)
            sum_mask = torch.clamp(mask.sum(1), min=1e-9)
            mean_pooled = sum_embeddings / sum_mask

            normalized_mean = F.normalize(mean_pooled, p=2, dim=1)

            embedding_list.append(normalized_mean.cpu())

    return torch.cat(embedding_list, dim=0)


dense_embeddings = extract_dense_embeddings(df['title'].tolist(), batch_size=64)


In [ ]:

def evaluate_embeddings_threshold(embeddings, df, threshold, chunk_size=1024):
    preds = []
    total = len(embeddings)

    for i in range(0, total, chunk_size):
        chunk = embeddings[i:i+chunk_size].to(device)
        sim_matrix = torch.mm(chunk, embeddings.to(device).T)
        
        for row in sim_matrix:
            matched_indices =  torch.where(row >= threshold)[0].cpu().numpy()

            preds.append(df['posting_id'].iloc[matched_indices].tolist())

    return preds

transformer_thresholds = np.arange(.65, .9, .05)
transformer_f1_scores = []
for thresh in transformer_thresholds:
    preds = evaluate_embeddings_threshold(dense_embeddings, df, thresh)
    score = calc_mean_f1(df['target'], preds)
    transformer_f1_scores.append(score)

best_transformer_idx = int(np.argmax(transformer_f1_scores))
best_transformer_thresh = transformer_thresholds[best_transformer_idx]
print("Best Threshold: ", best_transformer_thresh)
print("Best F1 score: ", transformer_f1_scores[best_transformer_idx])

df['transformer_predictions'] = evaluate_embeddings_threshold(
    dense_embeddings, df, threshold=best_transformer_thresh
)
display(results_table) 

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(thresholds, tfidf_f1_scores, marker="x", color="blue")
plt.title("Distribution of f1 scores vs thresholds in TF-IDF")
plt.xlabel("thresholds")
plt.ylabel("Mean F1 scores")
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(transformer_thresholds, transformer_f1_scores, marker="x", color="blue")
plt.title("Distribution of f1 scores vs thresholds in transformers")
plt.xlabel("Thresholds")
plt.ylabel("Mean F1 scores")
plt.grid(True)


plt.tight_layout()
plt.show()

results_table = pd.DataFrame({
    'Experiment': ['Baseline (TF-IDF)', 'Experiment 1 (MiniLM Transformer)'],
    'Representation': ['Character N-Grams (3-5)', 'Dense Semantic Vectors (384-d)'],
    'Similarity': ['Cosine', 'Cosine (Dot Product)'],
    'Best Threshold': [round(best_tfidf_thresh, 2), round(best_transformer_thresh, 2)],
    'Best F1 Score': [round(tfidf_f1_scores[best_tfidf_idx], 4), round(transformer_f1_scores[best_transformer_idx], 4)]
})

In [ ]:
found_fp = False

for idx in range(len(df)):
    true_set = set(df['target'].iloc[idx])
    pred_set = set(df['transformer_predictions'].iloc[idx])

    false_positives = list(pred_set - true_set)
    if false_positives:
        anchor_title= df['title'].iloc[idx]
        fp_title = df[df['posting_id'] == false_positives[0]]['title'].iloc[0]

        print("Anchor title: ", anchor_title)
        print("Matched title: ", fp_title)
        found_fp=True
        break

found_fn = False

for idx in range(len(df)):
    true_set = set(df['target'].iloc[idx])
    pred_set = set(df['tfidf_predictions'].iloc[idx])

    false_negs = list(true_set - pred_set)
    if false_negs:
        anchor_title = df["title"].iloc['idx']
        fn_title = df[df['posting_id'] == false_negs[0]]['title']

        print("Anchor title: ", anchor_title)
        print("Missing title: ", fn_title)
        found_fn=False
        break
    
